In [12]:
import json
import os

In [13]:
image_dir = 'E:\\Research\simplify-me\simplify_me_dataset\images'
meta_path = 'E:\\Research\simplify-me\simplify_me_dataset\meta.json'
environment = 'local'

REGISTRY = {
    'local': {
        'image_dir': 'E:\\Research\simplify-me\simplify_me_dataset\images',
        'meta_path': 'E:\\Research\simplify-me\simplify_me_dataset\meta.json',
        'environment': 'local',
    },
    'cc': {
        'image_dir': '/home/pranon/scratch/def-tahmedge/simplify-me-dataset/simplify-me/compressed_images',
        'meta_path': 'E:\\Research\simplify-me\simplify_me_dataset\meta.json',
        'environment': 'cc',
    }
}

In [14]:
def load_data():
    with open(meta_path, 'r', encoding='utf-8') as f:
        return json.load(f)

In [18]:
def get_and_preprocess_data(data, split, env):
    processed_data = []

    for item in data[split]:
        processed_data.append({
            'id': str(item['id']),
            'benchmark': item['benchmark'],
            'input': "<image>",
            'images': [os.path.join(REGISTRY[env]['image_dir'], item['benchmark'],
                                    item['image']['file_name'])],
            'instruction': 'Describe the image in simple terms',
            'chosen': item['dpo']['accepted']['caption'],
            'rejected': item['dpo']['rejected']['caption'],
        })

    filename = f'../data/{split}-{env}.json'

    with open(filename, 'w', encoding='utf-8') as fp:
        json.dump(processed_data, fp, indent=4, ensure_ascii=False)

    return filename

    # with open('./data/train.jsonl', 'w', encoding='utf-8') as f:
    #     for item in test_data:
    #         f.write(json.dumps(item, ensure_ascii=False) + '\n')

In [19]:
def get_dataset_info(env, split, filename):
    return {
        f'sm-{env}-{split}': {
            "file_name": filename,
            "ranking": True,
            "columns": {
                "prompt": "instruction",
                "query": "input",
                "chosen": "chosen",
                "rejected": "rejected",
                "images": "images"
            }
        },
    }

In [21]:
raw_data = load_data()

dataset_info = {}

for key in REGISTRY.keys():
    fn = get_and_preprocess_data(raw_data, 'train', key)
    ds_info = get_dataset_info(key, 'train', fn)

    dataset_info.update(ds_info)

    fn = get_and_preprocess_data(raw_data, 'test', key)
    ds_info = get_dataset_info(key, 'test', fn)
    dataset_info.update(ds_info)

with open('../data/dataset_info.json', 'w', encoding='utf-8') as f:
    json.dump(dataset_info, f, indent=4, ensure_ascii=False)